# Agentic Enterprise Architecture

**Level:** Advanced · **Time:** 90 min

Operating a governed ecosystem of agents requires an Enterprise Control Plane.

In this comprehensive notebook, we will simulate 4 critical enterprise architectural patterns:
1. **The Registry Gatekeeper:** Rejecting agents that lack required metadata (like ownership or risk tiers).
2. **Tool Squatting Defense:** Blocking malicious developers from registering fake tools using cryptographic provenance.
3. **Delegated Identity (IAM):** Why agents must use scoped JWTs instead of blanket API keys.
4. **FinOps Ceilings:** Automatically killing an agent loop when it exceeds its allocated financial budget.

---
## Pattern 1: The Registry Gatekeeper

Developers cannot deploy "Shadow Agents". All agents must be registered with the Enterprise Control Plane. If they lack an owner, a risk tier, or an evaluation score, they are rejected.

In [ ]:
def enterprise_registry_register_agent(agent_manifest: dict):
    print(f"[Registry] Attempting to register agent: {agent_manifest.get('name', 'Unknown')}")
    
    required_fields = ["name", "owner", "risk_tier", "eval_score"]
    for field in required_fields:
        if field not in agent_manifest:
            print(f"🚨 [Registry] REJECTED: Missing required field '{field}'. Shadow agents are not allowed in production.")
            return False
            
    print("[Registry] ✅ SUCCESS: Agent registered and approved for Orchestrator deployment.")
    return True

print("--- Scenario A: A Developer tries to deploy a quick hack script ---")
rogue_agent = {"name": "Quick_Data_Scraper"}
enterprise_registry_register_agent(rogue_agent)

print("\n--- Scenario B: A compliant engineering team registers an agent ---")
compliant_agent = {
    "name": "HR_Vacation_Summarizer",
    "owner": "team-hr-eng",
    "risk_tier": "low",
    "eval_score": 0.95
}
enterprise_registry_register_agent(compliant_agent)


---
## Pattern 2: Tool Squatting Defense

If anyone can publish a tool named `billing.refund_stripe`, an attacker can steal payloads. The registry must enforce cryptographic provenance.

In [ ]:
# Simulated Registry State
registered_namespaces = {
    "billing.refund_stripe": "pubkey_billing_team_x912"
}

def register_tool(namespace: str, signature: str):
    print(f"[Registry] Registration request for tool namespace: {namespace}")
    
    if namespace in registered_namespaces:
        # Enforce Cryptographic Provenance
        if signature != registered_namespaces[namespace]:
            print("🚨 [Registry] SECURITY ALERT: Tool Squatting Attempt Detected!")
            print(f"The signature '{signature}' does not match the official owner of '{namespace}'.")
            return False
            
    print(f"[Registry] ✅ SUCCESS: Tool {namespace} registered.")
    return True

print("--- Scenario A: Attacker tries to hijack the billing tool ---")
register_tool("billing.refund_stripe", "pubkey_attacker_hacker123")

print("\n--- Scenario B: The real billing team updates their tool ---")
register_tool("billing.refund_stripe", "pubkey_billing_team_x912")


---
## Pattern 3: Delegated Identity (IAM)

Agents must NEVER use a human's blanket API key. They must request a narrow-scope, time-bound JWT from the IAM provider.

In [ ]:
def mock_iam_provider_request_token(agent_id, requested_scope):
    print(f"[IAM] Agent '{agent_id}' is requesting scope '{requested_scope}'")
    # The IAM provider issues a narrow JWT
    return {"sub": agent_id, "scope": requested_scope, "exp": "15_mins"}

def mock_database_server(jwt_token, action):
    print(f"[Database] Received request to execute: '{action}'")
    
    if "read" in action and jwt_token["scope"] != "read_only":
         print("🚨 [Database] REJECTED: 403 Forbidden. Agent lacks read scope.")
         return False
    if "delete" in action and jwt_token["scope"] != "admin":
         print("🚨 [Database] REJECTED: 403 Forbidden. Agent lacks admin scope for deletion.")
         return False
         
    print("[Database] ✅ SUCCESS: 200 OK. Action executed.")
    return True

print("--- Scenario: A read-only agent gets hijacked and tries to delete data ---")
# 1. Orchestrator gets a narrow token for the agent
agent_token = mock_iam_provider_request_token("hr_agent_v1", "read_only")

# 2. The agent is hijacked (Prompt Injection) and tries to drop tables
mock_database_server(agent_token, "delete_all_employee_records")
print("[System] The blast radius was contained because the agent did not inherit human admin rights.")


---
## Pattern 4: FinOps Ceilings & Budgeting

Agents left unchecked can get into infinite loops and burn thousands of dollars in API credits. The Enterprise Orchestrator must enforce a strict budget ceiling per task.

In [ ]:
import time

TASK_BUDGET_USD = 0.50
COST_PER_LLM_CALL = 0.15

def orchestrator_run_agent_loop():
    current_spend = 0.0
    loop_count = 1
    
    print(f"[Orchestrator] Starting task. Budget ceiling: ${TASK_BUDGET_USD:.2f}")
    
    while True:
        print(f"\n  > [Agent] Running step {loop_count}...")
        time.sleep(0.5) # Simulate LLM call
        
        current_spend += COST_PER_LLM_CALL
        print(f"  > [FinOps] Spend updated: ${current_spend:.2f}")
        
        # 🚨 FinOps Enforcement 🚨
        if current_spend >= TASK_BUDGET_USD:
            print(f"\n🚨 [Orchestrator] FINOPS ALERT: Task budget of ${TASK_BUDGET_USD:.2f} exceeded.")
            print("[Orchestrator] FORCE KILLING AGENT LOOP to prevent runaway spend.")
            break
            
        loop_count += 1

orchestrator_run_agent_loop()
